# LELA60341: Research Methods in Computational and Corpus Linguistics Week 10

## A `LogisticRegression` class (2): evaluation

Last week you wrote a `LogisticRegression` class that can train on texts and make predictions. This week you finish it, by adding the evaluation that you did step by step in the LELA60331 Week 9 seminar:

```python
model = LogisticRegression(vocab_size=5000, lr=0.4, n_epochs=500)
model.train(train_texts, train_labels)
print(model.evaluate(test_texts, test_labels))
```

| Part | Topic |
|---|---|
| 0 | Set-up: the data, `TextEncoder`, and the helper functions |
| 1 | Adding `evaluate` and `top_words` to the class |
| 2 | Evaluating on the test set |
| 3 | Inspecting the weights |
| 4 | A better tokeniser, and two models side by side |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 0 · Set-up

Run the next three cells. They download the reviews, and set up the data, the `TextEncoder` class from Week 8, and the three helper functions from Week 9 (`sigmoid`, `sparse_mat_vec` and `sparse_transpose_mat_vec`).

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def load_reviews(path):
    texts = []
    labels = []
    with open(path) as fp:
        lines = fp.readlines()
    for line in lines[1:]:              # skip the header line
        fields = line.rstrip().split("\t")
        texts.append(fields[0])
        labels.append(int(fields[1] == "positive"))
    return texts, labels

texts, labels = load_reviews("yelp_reviews.txt")

random.seed(1)                          # the same split as in Week 8 and in LELA60331
train_ints = random.sample(range(len(texts)), 8000)
train_set = set(train_ints)
test_ints = [i for i in range(len(texts)) if i not in train_set]
train_texts = [texts[i] for i in train_ints]
train_labels = [labels[i] for i in train_ints]
test_texts = [texts[i] for i in test_ints]
test_labels = [labels[i] for i in test_ints]


class TextEncoder:
    """Turn texts into sparse one-hot rows over a vocabulary of the most frequent words."""

    def __init__(self, vocab_size=5000):
        self._vocab_size = vocab_size
        self._type_list = []
        self._word_to_index = {}

    def tokenise(self, text):
        """Split a text into a list of tokens."""
        return text.split()

    def fit_vocabulary(self, texts):
        """Build the vocabulary from the most frequent tokens in texts."""
        counts = {}
        for text in texts:
            for word in self.tokenise(text):
                counts[word] = counts.get(word, 0) + 1
        pairs = sorted([(count, word) for word, count in counts.items()], reverse=True)
        self._type_list = [word for count, word in pairs[:self._vocab_size]]
        self._word_to_index = {}
        for j in range(len(self._type_list)):
            self._word_to_index[self._type_list[j]] = j

    def __len__(self):
        return len(self._type_list)

    def __getitem__(self, j):
        return self._type_list[j]

    def index(self, word):
        """The column of word, or None if it is not in the vocabulary."""
        return self._word_to_index.get(word)

    def one_hot_encode(self, texts):
        """A list with one sorted list of columns per text."""
        M = []
        for text in texts:
            row = set()
            for word in self.tokenise(text):
                if word in self._word_to_index:
                    row.add(self._word_to_index[word])
            M.append(sorted(row))
        return M

    def decode(self, row):
        """The words for the columns in a sparse row."""
        return [self[j] for j in row]

print(len(train_texts), "training and", len(test_texts), "test reviews")

In [ ]:
def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    else:
        return math.exp(z) / (1 + math.exp(z))      # the same value, without calculating math.exp(-z)

def sparse_mat_vec(M, vector):
    result = []
    for row in M:
        total = 0
        for j in row:
            total += vector[j]
        result.append(total)
    return result

def sparse_transpose_mat_vec(M, vector, num_columns):
    result = [0] * num_columns
    for i in range(len(M)):
        for j in M[i]:
            result[j] += vector[i]
    return result

---
# Part 1 · Adding `evaluate` and `top_words`

The cell in Exercise 1 holds your `LogisticRegression` class from Week 9, with two new methods at the end. Adding methods to an existing class like this is how most classes grow: the code that already uses the class keeps working, because none of the old methods change.

**`evaluate(texts, labels)`** measures how good the model's predictions are, as in LELA60331 Week 9, Activities 4–5. It predicts the class of every text, counts the correct predictions, the true positives, the false positives and the false negatives, and returns a **dict** (Week 2) with three entries: `"accuracy"`, `"precision"` and `"recall"`. Reminder:

* **precision** = true positives / (true positives + false positives): of the texts the model *called* positive, how many really were?
* **recall** = true positives / (true positives + false negatives): of the texts that really *were* positive, how many did the model find?

**`top_words(n)`** returns the words that matter most to the model, as in LELA60331 Week 9, Activity 7. Each weight belongs to one word: words with **high** weights push the model towards *positive*, and words with **low** (very negative) weights push it towards *negative*. The method sorts `(weight, word)` pairs from the lowest weight to the highest, and returns two lists: the `n` words with the lowest weights, and the `n` with the highest (highest first). *Hint:* the object's own `[]` gives the word for column `j`: `self[j]`. The slice `[::-1]` reverses a list.

### 🏋️ Exercise 1 · Practice — Complete `evaluate` and `top_words`

Fill in the blanks 1–5. The tests underneath use the tiny corpus from Week 9.

In [ ]:
class LogisticRegression(TextEncoder):
    """Logistic regression for classifying texts, with one-hot word features."""

    def __init__(self, vocab_size=5000, lr=0.4, n_epochs=500, seed=2):
        super().__init__(vocab_size)
        self._lr = lr
        self._n_epochs = n_epochs
        self._seed = seed
        self._weights = []
        self._bias = 0
        self._loss_history = []

    # ---------------------------------------------------------------- prediction
    def predict_proba(self, texts):
        """The probability that each text is positive."""
        M = self.one_hot_encode(texts)
        return [sigmoid(value + self._bias) for value in sparse_mat_vec(M, self._weights)]

    def predict(self, texts):
        """The predicted class (1 or 0) of each text."""
        return [int(p > 0.5) for p in self.predict_proba(texts)]

    # ---------------------------------------------------------------- training
    def average_loss(self, q, labels):
        eps = 0.00001          # a tiny number, so that we never take the log of 0
        losses = []
        for k in range(len(labels)):
            if labels[k] == 1:
                losses.append(-math.log2(q[k] + eps))
            else:
                losses.append(-math.log2(1 - q[k] + eps))
        return sum(losses) / len(losses)

    def train(self, texts, labels, verbose=False):
        """Learn the weights and bias from texts and their labels (1 or 0)."""
        if len(self) == 0:                       # no vocabulary yet
            self.fit_vocabulary(texts)
        M = self.one_hot_encode(texts)
        num_features = len(self)
        num_samples = len(labels)

        random.seed(self._seed)                  # random starting weights, as in LELA60331 Week 5
        self._weights = [random.random() for j in range(num_features)]
        self._bias = random.random()
        self._loss_history = []

        for epoch in range(self._n_epochs):
            q = [sigmoid(value + self._bias) for value in sparse_mat_vec(M, self._weights)]
            self._loss_history.append(self.average_loss(q, labels))
            # errors and gradients (LELA60331 Week 7, Activity 1, with sparse_transpose_mat_vec)
            errors = [q[k] - labels[k] for k in range(num_samples)]
            dw = [value / num_samples for value in sparse_transpose_mat_vec(M, errors, num_features)]
            db = sum(errors) / num_samples
            # the update
            self._weights = [self._weights[j] - self._lr * dw[j] for j in range(num_features)]
            self._bias = self._bias - self._lr * db
            if verbose and epoch % 50 == 0:
                print("epoch", epoch, " loss", round(self._loss_history[-1], 3))

    def loss_history(self):
        return self._loss_history

    # ---------------------------------------------------------------- evaluation (new this week)
    def evaluate(self, texts, labels):
        """A dict with the accuracy, precision and recall of the model on texts."""
        predicted = ???                           # 1.
        correct = 0
        tp = 0
        fp = 0
        fn = 0
        for k in range(len(labels)):
            if predicted[k] == labels[k]:
                correct += 1
            if predicted[k] == 1 and labels[k] == 1:
                tp += 1
            if ???:              # 2. a false positive
                fp += 1
            if ???:              #    a false negative
                fn += 1
        return {"accuracy": ???,                # 3.
                "precision": ???,
                "recall": ???}

    def top_words(self, n):
        """The n words with the lowest weights, and the n with the highest (highest first)."""
        pairs = ???   # 4.
        lowest = [word for weight, word in pairs[:n]]
        highest = ???                     # 5.
        return lowest, highest


tiny_texts = ["great food", "great service", "lovely food great staff", "lovely place",
              "awful food", "awful service", "rude staff awful place", "rude service"]
tiny_labels = [1, 1, 1, 1, 0, 0, 0, 0]

tiny_model = LogisticRegression(vocab_size=10, lr=0.5, n_epochs=200)
tiny_model.train(tiny_texts, tiny_labels)
print(tiny_model.evaluate(tiny_texts, tiny_labels))
print(tiny_model.evaluate(["great food", "awful food", "great service"], [1, 1, 0]))
print(tiny_model.top_words(2))
# you should get: all three 1.0; then 1/3, 0.5 and 0.5; then (['awful', 'rude'], ['great', 'lovely'])

In [ ]:
# ✅ Check
check("1–3. evaluate when all are right", lambda: tiny_model.evaluate(tiny_texts, tiny_labels) == {"accuracy": 1.0, "precision": 1.0, "recall": 1.0})
check("1–3. evaluate when some are wrong", lambda: tiny_model.evaluate(["great food", "awful food", "great service"], [1, 1, 0]) == {"accuracy": 1/3, "precision": 0.5, "recall": 0.5})
check("4–5. top_words", lambda: tiny_model.top_words(2) == (["awful", "rude"], ["great", "lovely"]))
check("4–5. top_words with all the words", lambda: sorted(tiny_model.top_words(8)[0]) == sorted(tiny_model.decode(range(8))))
check("the Week 9 methods still work", lambda: tiny_model.predict(["great place", "awful staff"]) == [1, 0])

---
# Part 2 · Evaluating on the test set

### 🏋️ Exercise 2 · Practice — Test and training performance

1. Make `model` with the same settings as in Week 9 (5000 words, a learning rate of 0.4, 500 epochs), build its vocabulary from all of `texts`, and train it on the training reviews. Training takes about a minute.
2. `test_results` and `train_results`: the results of evaluating it on the test set and on the training set

In [ ]:
model = ???   # 1.
???
???

test_results = ???            # 2.
train_results = ???
print("test:    ", test_results)
print("training:", train_results)

In [ ]:
# ✅ Check
check("1. final loss 0.598", lambda: abs(model.loss_history()[-1] - 0.598) < 0.001)
check("2. test accuracy 0.82, precision 0.807, recall 0.829",
      lambda: abs(test_results["accuracy"] - 0.82) < 0.001 and abs(test_results["precision"] - 0.807) < 0.001 and abs(test_results["recall"] - 0.829) < 0.001)
check("2. training precision 0.857, recall 0.862",
      lambda: abs(train_results["precision"] - 0.857) < 0.001 and abs(train_results["recall"] - 0.862) < 0.001)

These are the same numbers as in the LELA60331 Week 9 seminar. Compare the two pieces of code: what does the class do for you that the seminar's cells made you do by hand?

As in the seminar, the model does better on the training reviews than on the test reviews: it has **overfitted** a little, learning some things that are true of the particular reviews it was trained on, but that don't hold in general.

---
# Part 3 · Inspecting the weights

### 🏋️ Exercise 3 · Practice — The most important words

1. `lowest` and `highest`: the 15 words with the lowest and the highest weights. *Hint:* `top_words` returns two lists, so unpack them (Week 4).
2. `lowest_100` and `highest_100`: the same for the 100 lowest and highest
3. `not_words`: the tokens in `lowest_100` and `highest_100` that are **not** made only of letters. *Hint:* `isalpha()` (Week 2).

Look through the lists. Which words are about sentiment in general? Which suggest **overfitting**: that they tell the model about the particular reviews it was trained on (LELA60331 Week 9, Activity 8)? Are the tokens in `not_words` useful to the model, or noise?

In [ ]:
lowest, highest = ???                    # 1.
print(lowest)
print(highest)

lowest_100, highest_100 = ???           # 2.
not_words = ???   # 3.
print(not_words)

In [ ]:
# ✅ Check
check("1. lowest", lambda: lowest[:3] == ["?", "no", "not"] and len(lowest) == 15)
check("1. highest", lambda: highest[:3] == ["great", "definitely", "excellent"] and len(highest) == 15)
check("2. 100 of each", lambda: len(lowest_100) == 100 and len(highest_100) == 100 and lowest_100[:15] == lowest)
check("3. not_words", lambda: not_words == ["?", "!"])

*Write your observations here.*

---
# Part 4 · A better tokeniser, and two models side by side

Many of the 5000 tokens in the vocabulary aren't really words: punctuation, numbers, and `n` (what is left of the line breaks in the original reviews). Run `print([model[j] for j in range(len(model)) if not model[j].isalpha()][:40])` to see some. Exercise 3 showed that two of them, `?` and `!`, are among the most useful tokens of all. Would the model be better without the rest?

In the LELA60331 Week 9 seminar you changed the tokenisation cell and re-ran everything, losing the first model.

With classes, you can write a subclass that overrides `tokenise`, as you did with `StopwordEncoder` in Week 8, and keep **both** models to compare.

### 🏋️ Exercise 4 · Practice — `CleanLogisticRegression`

1. Complete `tokenise` in `CleanLogisticRegression`, a subclass of `LogisticRegression`. It should call the parent's `tokenise` and keep only the tokens that are made of letters (`isalpha()`, from Week 2) and are not `"n"`. (That throws away `!` and `?` too. Do you think that is a good idea? You can try other rules afterwards.)
2. Make `clean_model` with the same settings as `model`, build its vocabulary from all the texts and train it. Nothing else needs changing: `fit_vocabulary`, `one_hot_encode`, `train`, `predict_proba` and `evaluate` all end up calling `self.tokenise`.
3. `clean_results`: its results on the test set. Is it better or worse than `model`? Look at its top words: what has changed? Can you explain the difference? (Think about Exercise 3.) Try a tokeniser that keeps `!` and `?`.

In [ ]:
class CleanLogisticRegression(LogisticRegression):
    """Logistic regression that only uses tokens made of letters."""

    def tokenise(self, text):
        return ???    # 1.

print(CleanLogisticRegression().tokenise("i paid $ 20 for 2 pizzas ! n nthe best ."))

clean_model = ???      # 2.
???
???

clean_results = ???                    # 3.
print("naive tokens:", test_results)
print("clean tokens:", clean_results)
print(clean_model.top_words(15))

In [ ]:
# ✅ Check
check("1. tokenise", lambda: CleanLogisticRegression().tokenise("i paid $ 20 for 2 pizzas ! n nthe best .") == ["i", "paid", "for", "pizzas", "nthe", "best"])
check("2. a LogisticRegression too", lambda: isinstance(clean_model, LogisticRegression) and len(clean_model) == 5000)
check("2. no punctuation or numbers in its vocabulary", lambda: clean_model.index(".") is None and clean_model.index("n") is None)
check("3. clean_results", lambda: set(clean_results) == {"accuracy", "precision", "recall"})
check("the first model is unchanged", lambda: model.index(".") == 0 and abs(model.evaluate(test_texts, test_labels)["accuracy"] - 0.82) < 0.001)

Look at the token `nthe` in the output of `tokenise`: it is a line break stuck to the next word. ⭐ **Challenge (optional):** write a tokeniser that also fixes tokens like this, by removing an `n` from the start of any token that isn't a word on its own. How can you tell which tokens to fix? (*Hint:* is `nthe` in the vocabulary of `model`? Is `the`?) Compare its results with the other two models.

In [ ]:
# ⭐ Challenge

---
## ✏️ Summary

You can now:

* add new methods to an existing class without changing the code that already uses it
* return several results as a **dict**
* evaluate a classifier on a test set, and recognise overfitting
* inspect what a model has learned from its weights
* change one method (`tokenise`) in a subclass, and keep two models side by side to compare them

Over Weeks 8, 9 and 10 you have written, in about 100 lines, the complete review classifier from the LELA60331 seminars. Libraries such as scikit-learn are organised in exactly this way: a model is an object with methods like `fit` and `predict`.